# Data Cleaning Notebook

用于 `my_ml_backend` 的数据清洗与抽样检查。

In [ ]:
import os

import pandas as pd
from sqlalchemy import create_engine

DORIS_HOST = os.environ.get("DORIS_HOST", "127.0.0.1")
DORIS_PORT = os.environ.get("DORIS_PORT", "9030")
DORIS_USER = os.environ.get("DORIS_USER", "root")
DORIS_PASSWORD = os.environ.get("DORIS_PASSWORD", "")
DORIS_DATABASE = os.environ.get("DORIS_DATABASE", "ysmy")

engine = create_engine(
    f"mysql+pymysql://{DORIS_USER}:{DORIS_PASSWORD}@{DORIS_HOST}:{DORIS_PORT}/{DORIS_DATABASE}?charset=utf8mb4"
)

In [17]:
df1 = pd.read_sql_query(f"""
    SELECT DISTINCT buyersNickname AS buyersNickname
    FROM furniture_trade__trade_order_line_s_o
    WHERE `updatedAt` >= '2025-09-01'
""", engine)

df1.head()

,buyersNickname
0,小女人030906
1,nightelves19840505
2,tb5285685_2013
3,雨虹
4,13241173458wu


In [18]:
df2 = pd.read_sql_query(f"""
    SELECT DISTINCT expressNumber AS expressNumber
    FROM furniture_tms_busi__express_detail
    WHERE `updatedAt` >= '2025-09-01'
""", engine)

df2.head()

,expressNumber
0,S66344230091
1,301639652048
2,760196161240
3,S66343862217
4,S66344200407


In [26]:
df2["expressNumber"] = (
    df2["expressNumber"]
    .astype(str)
    .str.replace(r"[^0-9a-zA-Z-]", "", regex=True)
)

# 长度过滤
df2 = df2[
    df2["expressNumber"].str.len().between(7, 30)
]

df2["len"] = df2["expressNumber"].str.len()

length_stats = (
    df2.groupby("len")
      .agg(
          count=("expressNumber", "size"),
          samples=("expressNumber", lambda x: list(x.head(10)))
      )
      .reset_index()
)

length_stats

,len,count,samples
0,7,6,"[1111111, 0000836, 1013236, 2277358, 2277503, ..."
1,8,4,"[25447854, 00003832, 13318310, 12980843]"
2,9,48,"[238481019, 298433342, 233700214, 955464003, 9..."
3,10,5,"[2003145109, 8981HZD002, 2038684845, 808046344..."
4,11,226749,"[92771230591, 92771240203, 92771240262, 821818..."
5,12,860978,"[S66344230091, 301639652048, 760196161240, S66..."
6,13,4,"[8000010541242, DZ-K35E05156A, MSF0009063869, ..."
7,14,4470,"[78997363369539, 78951334298769, 7900303802638..."
8,15,673281,"[434810334365005, 320737296923255, SF329718268..."
9,16,6,"[2DPK364962240653, JDVE16954282961C, DPK364949..."


In [30]:
df1["len"] = df1["buyersNickname"].str.len()

length_stats1 = (
    df1.groupby("len")
      .agg(
          count=("buyersNickname", "size"),
          samples=("buyersNickname", lambda x: list(x.head(10)))
      )
      .reset_index()
)

length_stats1

,len,count,samples
0,1,218,"[依, 维, a, 藤, 薇, F, ๑, 苗, 夏, M]"
1,2,2226,"[雨虹, 季杨, 顾成, 緢绘, 砂锅, 璟筠, 言听, 嗨皮, 虔人, 启穗]"
2,3,10225,"[朱平亚, 张烛远, 柳拂衣, 高新武, 十二婷, 杜肚子, B2B, 凉风声, 边立强, ..."
3,4,14401,"[清愁浅浅, 姚姚当当, 普若年华, 小手公园, 浅香雅美, 尤溪米米, 安安聆听, 中华玉..."
4,5,16172,"[上帝的泪f, 愿我心长眠, 归零282, 疯狂李小莫, 绑绑糖lp, 晨晨gxc, 月夜江..."
...,...,...,...
93,95,6555,[AAEc5t2oAJmLtqP7Ur3vjM0u&AAEX5t2oAJmLtqP7Ur2m...
94,96,6,[iepSzAGzKAvA3ReKkuWswdyX1DMW0CdF3Hy+Eyf+Nx3yq...
95,97,6,[9I7v18wU5DD0Qw4o+ItWN6JZ8IAsWoabHzdcBCyo2Yt8B...
96,98,2,[s350IKLYwXD4iXBoVb0MXStv5P2LB3F6lQENKwCCYyRPN...
